# Notebook 06: Explainability

XAI analysis of **HeartBERT LoRA r=8** (best-performing transformer, test AUC 0.82)
on 15 PTB-XL test records (3 per superclass, fixed seed 42).

**Method:** Last-layer RoBERTa attention weights (CLS token attending to letter tokens)
overlaid on the Lead II waveform, plus a Qwen3-0.6B clinical narrative per record.

**Outputs:** `results/06_explainability/narratives.json` and `figures/*.png`

In [1]:
import sys, os, warnings, json, gc
from pathlib import Path
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import matplotlib.pyplot as plt
import torch
import wfdb

from src.utils.config import CFG
from src.preprocessing.label_utils import load_all_labels, SUPERCLASSES
from src.explainability.llm import load_qwen3, build_ecg_prompt, generate_explanation, safe_print

np.random.seed(42)
torch.manual_seed(42)

DATA_PATH    = CFG['data']['path']
RESULTS_PATH = CFG['paths']['results']
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

LEAD_IDX   = 1
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

OUT_DIR = Path(RESULTS_PATH) / '06_explainability'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Output : {OUT_DIR}')

Device : cuda
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU
Output : D:\GitHub\biosignal-xai\results\06_explainability


## 2. Data

Sample 3 records per superclass from the PTB-XL test fold (fold 10).
Seed 42 is fixed so the selection is reproducible.

In [2]:
Y       = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')
test_df = Y[Y.strat_fold == 10].reset_index(drop=True)

np.random.seed(42)
demo_records = {}
for cls_idx, cls in enumerate(SUPERCLASSES):
    mask       = test_df['label_vec'].apply(lambda v: v[cls_idx] == 1.0)
    candidates = test_df[mask]
    chosen     = candidates.sample(n=3, random_state=42)
    demo_records[cls] = chosen.to_dict('records')

for cls in SUPERCLASSES:
    assert len(demo_records[cls]) == 3, f'Expected 3 records for {cls}'

print('Demo records per class:')
for cls, rows in demo_records.items():
    print(f'  {cls}: {[r["filename_lr"] for r in rows]}')

Records with valid labels: 21375
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5108 (23.9%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
Demo records per class:
  NORM: ['records100/10000/10215_lr', 'records100/05000/05771_lr', 'records100/05000/05916_lr']
  MI: ['records100/07000/07700_lr', 'records100/03000/03336_lr', 'records100/18000/18710_lr']
  STTC: ['records100/06000/06666_lr', 'records100/10000/10517_lr', 'records100/20000/20753_lr']
  CD: ['records100/19000/19848_lr', 'records100/03000/03486_lr', 'records100/10000/10283_lr']
  HYP: ['records100/20000/20211_lr', 'records100/16000/16061_lr', 'records100/14000/14201_lr']


## 3. Load HeartBERT LoRA r=8

In [3]:
from src.models.heartbert import HeartBERTClassifier

EXP   = 'heartbert_lora_r8'
model = HeartBERTClassifier(num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')
print(f'HeartBERT LoRA loaded from results/{EXP}/best_adapter')

Bayesiano/HeartBERT not available — loading roberta-base (same architecture).


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
HeartBERT LoRA loaded from results/heartbert_lora_r8/best_adapter


## 4. Load Qwen3-0.6B

First run downloads ~400 MB from HuggingFace. Subsequent runs use the local cache.

In [ ]:
qwen_model, qwen_tokenizer = load_qwen3('Qwen/Qwen2-0.5B-Instruct')

## 5. Helper functions

In [ ]:
def predict_single(model, x, device):
    """Run HeartBERT on a single (1000,) Lead II signal. Returns result dict."""
    logits = model.predict_logits(x[np.newaxis])           # (1, 5)
    probs  = torch.sigmoid(torch.tensor(logits))[0]        # (5,)
    preds  = [SUPERCLASSES[i] for i, p in enumerate(probs) if p.item() >= 0.5]
    if not preds:
        preds = [SUPERCLASSES[probs.argmax().item()]]
    return {
        'predicted_classes':   preds,
        'class_probabilities': {c: round(probs[i].item(), 4) for i, c in enumerate(SUPERCLASSES)},
        'confidence_score':    round(probs.max().item(), 4),
        'uncertainty':         None,
        'raw_logits':          logits[0].tolist(),
        'uncertainty_level':   'not computed',
    }


def plot_record(ax_ecg, ax_attn, signal, positions, weights, title):
    """Plot Lead II waveform with attention weight overlay."""
    t = np.arange(len(signal)) / 100.0
    ax_ecg.plot(t, signal, color='#1a5276', linewidth=0.8)
    ax_ecg.set_ylabel('Amplitude (mV)')
    ax_ecg.set_title(title, fontsize=9)
    ax_ecg.grid(True, alpha=0.3)
    ax_ecg.set_xlim(0, len(signal) / 100.0)

    t_attn = positions / 100.0
    ax_attn.fill_between(t_attn, weights, alpha=0.7, color='#e74c3c')
    ax_attn.axhline(0.5, color='#888', linewidth=0.8, linestyle='--')
    ax_attn.set_ylabel('Attention')
    ax_attn.set_xlabel('Time (s)')
    ax_attn.set_xlim(0, len(signal) / 100.0)
    ax_attn.set_ylim(0, 1.1)
    ax_attn.grid(True, alpha=0.3)


narratives = {cls: [] for cls in SUPERCLASSES}
print('Helper functions defined.')

## 6. NORM (Normal ECG)

Three normal ECG records. HeartBERT should assign high NORM probability
and low probability to pathological classes.

In [ ]:
CLASS = 'NORM'

for idx, row in enumerate(demo_records[CLASS]):
    sig, _ = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
    x = sig[:, LEAD_IDX].astype(np.float32)
    true_classes = [c for c, v in zip(SUPERCLASSES, row['label_vec']) if v == 1.0]

    result          = predict_single(model, x, device)
    positions, attn = model.get_attention_weights(x)
    prompt          = build_ecg_prompt(result, saliency=None,
                                       lead_names=LEAD_NAMES,
                                       true_classes=true_classes)
    narrative       = generate_explanation(prompt, qwen_model, qwen_tokenizer)

    fig, (ax_ecg, ax_attn) = plt.subplots(2, 1, figsize=(14, 5),
                                           gridspec_kw={'height_ratios': [3, 1]})
    title = (f'{CLASS} record {idx+1}  |  true: {true_classes}  |  '
             f'pred: {result["predicted_classes"]}  |  conf: {result["confidence_score"]:.2f}')
    plot_record(ax_ecg, ax_attn, x, positions, attn, title)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f'{CLASS.lower()}_{idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

    safe_print(f'--- Record {idx+1} narrative ---\n{narrative}\n')
    narratives[CLASS].append({
        'record':       row['filename_lr'],
        'true_classes': true_classes,
        'predicted':    result['predicted_classes'],
        'confidence':   result['confidence_score'],
        'narrative':    narrative,
    })

## 7. MI (Myocardial Infarction)

Three MI records. ST-segment and Q-wave changes are the key discriminating features.

In [ ]:
CLASS = 'MI'

for idx, row in enumerate(demo_records[CLASS]):
    sig, _ = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
    x = sig[:, LEAD_IDX].astype(np.float32)
    true_classes = [c for c, v in zip(SUPERCLASSES, row['label_vec']) if v == 1.0]

    result          = predict_single(model, x, device)
    positions, attn = model.get_attention_weights(x)
    prompt          = build_ecg_prompt(result, saliency=None,
                                       lead_names=LEAD_NAMES,
                                       true_classes=true_classes)
    narrative       = generate_explanation(prompt, qwen_model, qwen_tokenizer)

    fig, (ax_ecg, ax_attn) = plt.subplots(2, 1, figsize=(14, 5),
                                           gridspec_kw={'height_ratios': [3, 1]})
    title = (f'{CLASS} record {idx+1}  |  true: {true_classes}  |  '
             f'pred: {result["predicted_classes"]}  |  conf: {result["confidence_score"]:.2f}')
    plot_record(ax_ecg, ax_attn, x, positions, attn, title)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f'{CLASS.lower()}_{idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

    safe_print(f'--- Record {idx+1} narrative ---\n{narrative}\n')
    narratives[CLASS].append({
        'record':       row['filename_lr'],
        'true_classes': true_classes,
        'predicted':    result['predicted_classes'],
        'confidence':   result['confidence_score'],
        'narrative':    narrative,
    })

## 8. STTC (ST/T-wave Change)

Three STTC records. T-wave inversions and ST-segment shifts are the primary markers.

In [ ]:
CLASS = 'STTC'

for idx, row in enumerate(demo_records[CLASS]):
    sig, _ = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
    x = sig[:, LEAD_IDX].astype(np.float32)
    true_classes = [c for c, v in zip(SUPERCLASSES, row['label_vec']) if v == 1.0]

    result          = predict_single(model, x, device)
    positions, attn = model.get_attention_weights(x)
    prompt          = build_ecg_prompt(result, saliency=None,
                                       lead_names=LEAD_NAMES,
                                       true_classes=true_classes)
    narrative       = generate_explanation(prompt, qwen_model, qwen_tokenizer)

    fig, (ax_ecg, ax_attn) = plt.subplots(2, 1, figsize=(14, 5),
                                           gridspec_kw={'height_ratios': [3, 1]})
    title = (f'{CLASS} record {idx+1}  |  true: {true_classes}  |  '
             f'pred: {result["predicted_classes"]}  |  conf: {result["confidence_score"]:.2f}')
    plot_record(ax_ecg, ax_attn, x, positions, attn, title)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f'{CLASS.lower()}_{idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

    safe_print(f'--- Record {idx+1} narrative ---\n{narrative}\n')
    narratives[CLASS].append({
        'record':       row['filename_lr'],
        'true_classes': true_classes,
        'predicted':    result['predicted_classes'],
        'confidence':   result['confidence_score'],
        'narrative':    narrative,
    })

## 9. CD (Conduction Disturbance)

Three CD records. Bundle branch blocks and AV blocks produce distinctive morphology changes.

In [ ]:
CLASS = 'CD'

for idx, row in enumerate(demo_records[CLASS]):
    sig, _ = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
    x = sig[:, LEAD_IDX].astype(np.float32)
    true_classes = [c for c, v in zip(SUPERCLASSES, row['label_vec']) if v == 1.0]

    result          = predict_single(model, x, device)
    positions, attn = model.get_attention_weights(x)
    prompt          = build_ecg_prompt(result, saliency=None,
                                       lead_names=LEAD_NAMES,
                                       true_classes=true_classes)
    narrative       = generate_explanation(prompt, qwen_model, qwen_tokenizer)

    fig, (ax_ecg, ax_attn) = plt.subplots(2, 1, figsize=(14, 5),
                                           gridspec_kw={'height_ratios': [3, 1]})
    title = (f'{CLASS} record {idx+1}  |  true: {true_classes}  |  '
             f'pred: {result["predicted_classes"]}  |  conf: {result["confidence_score"]:.2f}')
    plot_record(ax_ecg, ax_attn, x, positions, attn, title)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f'{CLASS.lower()}_{idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

    safe_print(f'--- Record {idx+1} narrative ---\n{narrative}\n')
    narratives[CLASS].append({
        'record':       row['filename_lr'],
        'true_classes': true_classes,
        'predicted':    result['predicted_classes'],
        'confidence':   result['confidence_score'],
        'narrative':    narrative,
    })

## 10. HYP (Hypertrophy)

Three HYP records. Increased QRS amplitude and axis deviation are the main indicators.

In [ ]:
CLASS = 'HYP'

for idx, row in enumerate(demo_records[CLASS]):
    sig, _ = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
    x = sig[:, LEAD_IDX].astype(np.float32)
    true_classes = [c for c, v in zip(SUPERCLASSES, row['label_vec']) if v == 1.0]

    result          = predict_single(model, x, device)
    positions, attn = model.get_attention_weights(x)
    prompt          = build_ecg_prompt(result, saliency=None,
                                       lead_names=LEAD_NAMES,
                                       true_classes=true_classes)
    narrative       = generate_explanation(prompt, qwen_model, qwen_tokenizer)

    fig, (ax_ecg, ax_attn) = plt.subplots(2, 1, figsize=(14, 5),
                                           gridspec_kw={'height_ratios': [3, 1]})
    title = (f'{CLASS} record {idx+1}  |  true: {true_classes}  |  '
             f'pred: {result["predicted_classes"]}  |  conf: {result["confidence_score"]:.2f}')
    plot_record(ax_ecg, ax_attn, x, positions, attn, title)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f'{CLASS.lower()}_{idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

    safe_print(f'--- Record {idx+1} narrative ---\n{narrative}\n')
    narratives[CLASS].append({
        'record':       row['filename_lr'],
        'true_classes': true_classes,
        'predicted':    result['predicted_classes'],
        'confidence':   result['confidence_score'],
        'narrative':    narrative,
    })

## 11. Save outputs

In [ ]:
with open(OUT_DIR / 'narratives.json', 'w') as f:
    json.dump(narratives, f, indent=2)

for cls, items in narratives.items():
    for item in items:
        assert len(item['narrative']) > 0, f'Empty narrative for {cls}: {item["record"]}'

print('Saved:')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(Path(RESULTS_PATH))}  ({p.stat().st_size / 1024:.1f} KB)')
print('All 15 narratives verified non-empty.')